# 🚗 Araç Tespiti ve YOLO11 Model Eğitimi (Uçtan Uca Pipeline)
### Bitirme Projesi - Otomatik Veri Analizi, Eğitim ve Kıyaslama

Bu notebook, Drive'daki yeni araç veri setini kullanarak **YOLO11** modelini eğitmek, başarı metriklerini ölçmek ve gerçek trafik fotoğrafı üzerinde eğitim öncesi vs. sonrası karşılaştırma görseli üretmek için **7 sıralı adımda** sadeleştirilmiştir.

👉 **Tüm hücreleri yukarıdan aşağıya sırayla (veya Çalışma zamanı -> Tümünü çalıştır diyerek) çalıştırmanız yeterlidir.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## Adım 1: Donanım Kontrolü ve Kütüphane Kurulumu
GPU durumunu kontrol edip Ultralytics (YOLO11) kütüphanesini kuruyoruz.


In [ ]:
import os
import torch

if torch.cuda.is_available():
    print(f'✅ Harika! GPU devrede: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL! Lütfen üst menüden: Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU seçip Kaydet deyin.')

!pip install -q ultralytics
print('✅ Ultralytics ve PyTorch ortamı hazır!')


## Adım 2: GitHub Proje Deposunu Güncelleme
Test fotoğraflarını ve proje ayarlarını çekiyoruz.


In [ ]:
import os

if not os.path.exists('federated-traffic-learning'):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git
    %cd federated-traffic-learning
else:
    %cd federated-traffic-learning
    !git pull

print('✅ Proje dizini hazır:', os.getcwd())


## Adım 3: Google Drive'dan Veri Setini Otomatik Yükleme ve Açma
Drive'a yüklediğiniz veri seti zip arşivini bulur ve yerel diske açar.


In [ ]:
import os
import glob
from google.colab import drive

# Google Drive'ı bağla
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Drive içinde veri seti zip dosyasını ara
target_names = ['vehicle-detection-clean.zip', 'ua_detrac_yolo.zip', '*vehicle*.zip', '*detrac*.zip']
found_zips = []
for pattern in target_names:
    matches = glob.glob(f'/content/drive/MyDrive/**/{pattern}', recursive=True)
    if matches:
        found_zips.extend(matches)
        break

extract_target = '/content/ua_detrac_dataset'

if found_zips:
    src_zip = found_zips[0]
    print(f'✅ Drive\'da veri seti bulundu: {src_zip}')
    print('📦 Arşiv açılıyor (Bu işlem dosya boyutuna göre 1-2 dakika sürebilir)...')
    !unzip -q -o "{src_zip}" -d "{extract_target}"
    print('✅ Veri seti başarıyla açıldı!')
elif os.path.exists(os.path.join(extract_target, 'train')) or glob.glob(f'{extract_target}/**/train', recursive=True):
    print('✅ Veri seti zaten /content/ua_detrac_dataset dizininde açılmış ve hazır!')
else:
    raise FileNotFoundError('Drive içerisinde veri seti zip dosyası bulunamadı! Lütfen dosyanın Drive ana dizininde olduğundan emin olun.')


## Adım 4: Veri Seti Detaylı Analizi ve data.yaml Yapılandırması 📊
Yeni veri setinin tüm sınıflarını (`car`, `motorcycle`, `bus`, `truck`), etiketli araç sayılarını ve oranlarını analiz edip `data.yaml` dosyasını oluşturur.


In [ ]:
import os
import glob
from collections import Counter
import yaml

# Kök dizini dinamik olarak tespit et
possible_roots = [
    '/content/ua_detrac_dataset/vehicle-detection-clean',
    '/content/ua_detrac_dataset'
]
dataset_root = None
for pr in possible_roots:
    if os.path.exists(os.path.join(pr, 'train')):
        dataset_root = pr
        break

if not dataset_root:
    found = glob.glob('/content/ua_detrac_dataset/**/train', recursive=True)
    if found:
        dataset_root = os.path.dirname(found[0])

if not dataset_root or not os.path.exists(dataset_root):
    raise FileNotFoundError('Veri seti klasörü bulunamadı! Lütfen Adım 3\'ü kontrol edin.')

# 1. data.yaml oluştur
yaml_content = f"""# Vehicle Detection YOLO Dataset Configuration
path: {dataset_root}
train: train/images
val: val/images
test: val/images

nc: 4
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
"""
yaml_path = '/content/ua_detrac_dataset/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip() + '\n')

# 2. Detaylı Sınıf ve Kutu Analizi
class_names_map = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}
print('=' * 68)
print('         📊 YENİ ARAÇ VERİ SETİ DETAYLI ANALİZ RAPORU')
print('=' * 68)
print(f'📁 Veri Seti Dizin Yolu: {dataset_root}\n')

total_overall_boxes = 0
overall_class_counts = Counter()

for split in ['train', 'val']:
    split_dir = os.path.join(dataset_root, split)
    if not os.path.exists(split_dir):
        continue
    img_dir = os.path.join(split_dir, 'images')
    lbl_dir = os.path.join(split_dir, 'labels')
    
    imgs = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png') + glob.glob(f'{img_dir}/*.jpeg')
    lbls = glob.glob(f'{lbl_dir}/*.txt')
    
    class_counts = Counter()
    split_boxes = 0
    for lf in lbls:
        try:
            with open(lf, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cls_id = int(float(parts[0]))
                        class_counts[cls_id] += 1
                        split_boxes += 1
        except:
            pass
            
    overall_class_counts.update(class_counts)
    total_overall_boxes += split_boxes
    
    print(f'📌 [{split.upper()} KÜMESİ]')
    print(f'   • Toplam Görsel Sayısı: {len(imgs):,}')
    print(f'   • Toplam Etiket Dosyası: {len(lbls):,}')
    print(f'   • Toplam Araç Kutusu (Bounding Box): {split_boxes:,}')
    print('   • Sınıf Dağılımı:')
    for cid in sorted(class_counts.keys()):
        cname = class_names_map.get(cid, f'class_{cid}')
        pct = (class_counts[cid] / split_boxes * 100) if split_boxes > 0 else 0.0
        print(f'      - ID {cid:02d} ({cname:<12}): {class_counts[cid]:>7,} adet  (%{pct:>5.1f})')
    print('-' * 68)

print('\n🎯 [GENEL TOPLAM]')
print(f'   • Toplam Etiketlenen Araç Sayısı: {total_overall_boxes:,}')
for cid in sorted(overall_class_counts.keys()):
    cname = class_names_map.get(cid, f'class_{cid}')
    pct = (overall_class_counts[cid] / total_overall_boxes * 100) if total_overall_boxes > 0 else 0.0
    print(f'   • Sınıf {cid} ({cname:<12}): {overall_class_counts[cid]:>8,} adet  (%{pct:>5.1f})')
print('=' * 68)
print(f'✅ data.yaml hazır: {yaml_path}')


## Adım 5: YOLO11 Modelini Eğitme (Training) 🚀
Modeli yeni zengin veri setimiz üzerinde T4 GPU hızlandırıcısı ile eğitiyoruz.


In [ ]:
import os
import torch
from ultralytics import YOLO

device_arg = 0 if torch.cuda.is_available() else 'cpu'
print(f'🚀 Eğitim başlatılıyor... Donanım: {device_arg} (GPU: {torch.cuda.is_available()})')

# YOLO11 Nano modelini yüklüyoruz
model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/ua_detrac_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=device_arg,
    project='runs/train',
    name='yolo11n_exp',
    patience=10,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


## Adım 6: Başarım Testi ve Gerçek Trafik Fotoğrafı Karşılaştırması 📸
Eğitilen modelin mAP skorlarını hesaplar ve test fotoğrafı üzerinde eğitim öncesi vs. sonrası tespit farkını yan yana görselleştirir.


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# 1. Test Kümesi Değerlendirmesi
metrics = model.val(data='/content/ua_detrac_dataset/data.yaml')
print('\n' + '=' * 50)
print('      MODEL BAŞARI METRİKLERİ')
print('=' * 50)
print(f"mAP@50:     {metrics.box.map50:.4f} (%{metrics.box.map50 * 100:.2f})")
print(f"mAP@50-95:  {metrics.box.map:.4f} (%{metrics.box.map * 100:.2f})")
print(f"Precision:  {metrics.box.mp:.4f} (%{metrics.box.mp * 100:.2f})")
print(f"Recall:     {metrics.box.mr:.4f} (%{metrics.box.mr * 100:.2f})")
print('=' * 50)

# 2. Gerçek Trafik Test Fotoğrafı Üzerinde Kıyaslama
custom_photo = 'test_samples/user_test_photo.jpg'
trained_weights = 'runs/train/yolo11n_exp/weights/best.pt'

if os.path.exists(custom_photo) and os.path.exists(trained_weights):
    print('\n📸 Özel test fotoğrafı kıyaslaması oluşturuluyor...')
    base_model = YOLO('yolo11n.pt')
    base_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='base_pred', exist_ok=True)
    
    trained_model = YOLO(trained_weights)
    trained_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='trained_pred', exist_ok=True)
    
    base_imgs = glob.glob('runs/detect/base_pred/*.*')
    trained_imgs = glob.glob('runs/detect/trained_pred/*.*')
    
    if base_imgs and trained_imgs:
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        axes[0].imshow(Image.open(base_imgs[0]))
        axes[0].set_title('Eğitim Öncesi: Base YOLO11n (Pretrained)', fontsize=15)
        axes[0].axis('off')
        
        axes[1].imshow(Image.open(trained_imgs[0]))
        axes[1].set_title('Eğitim Sonrası: Yeni Veriyle Eğitilmiş Model', fontsize=15)
        axes[1].axis('off')
        
        plt.tight_layout()
        out_comp = 'runs/train/yolo11n_exp/custom_test_comparison.jpg'
        plt.savefig(out_comp, bbox_inches='tight', dpi=150)
        print(f'✅ Karşılaştırma görseli kaydedildi: {out_comp}')
        plt.show()


## Adım 7: Sonuçları Paketleme ve Otomatik İndirme 📦
Eğitilen `best.pt` ağırlıklarını, başarı grafiklerini ve karşılaştırma görselini tek bir `yolo11_results.zip` dosyası haline getirip bilgisayarınıza indirir.


In [ ]:
import os
import shutil
from google.colab import files

exp_dir = 'runs/train/yolo11n_exp'
best_weights = f'{exp_dir}/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ Eğitilmiş en iyi model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', exp_dir)
files.download('yolo11_results.zip')
print('\n🎉 TEBRİKLER! Tüm işlemler başarıyla tamamlandı!')
print('👉 İnen "yolo11_results.zip" dosyasını masaüstündeki proje klasörünüze atıp asistanınıza bildirebilirsiniz.')
